# 04 · Retrieval and evaluation
Three retrieval methods over the indexes from 03, and the evaluation that decides between them:
- **Dense** (Chroma), **BM25**, and **hybrid** = Reciprocal Rank Fusion of the two (k=60, rank-based, no score mixing).
- Role and source filters are hard filters applied at search time. Answers only use evidence roles (textbook, note, web).
- **Reranker** (`BAAI/bge-reranker-base`, CPU) scores the top 20 candidates. Its sigmoid score feeds the abstain decision; reordering by it is optional and measured here.

Eval set: `data/eval/questions.csv` (DRAFT rows still need your check). Metrics: Recall@5, MRR, nDCG@5, per method and per question type, for three embedders.

Outputs: `data/eval/retrieval_results.json` (all metrics) and `data/eval/abstain_threshold.json` (used by 05).

## 1 · Imports and settings

In [1]:
import csv, hashlib, json, math, pickle, re, time
from collections import Counter, defaultdict
from pathlib import Path

import chromadb
import numpy as np
import requests
from sentence_transformers import CrossEncoder

DATA_DIR = Path("../data")
CHUNKS_FILE = DATA_DIR / "chunks" / "chunks.jsonl"
CHUNK_SETTINGS_FILE = DATA_DIR / "chunks" / "settings.json"
CACHE_DIR = DATA_DIR / "embed_cache"
CHROMA_DIR = DATA_DIR / "chroma"
BM25_FILE = DATA_DIR / "bm25" / "bm25.pkl"
EVAL_DIR = DATA_DIR / "eval"
QUESTIONS_FILE = EVAL_DIR / "questions.csv"

OLLAMA_URL = "http://localhost:11434"
EVIDENCE_ROLES = ["textbook", "note", "web"]
K = 5                      # results the answer step sees
CANDIDATES = 20            # pool each method returns for fusion / reranking
RRF_K = 60
RERANKER = "BAAI/bge-reranker-base"

QWEN_INSTRUCT = "Instruct: Given a student's question, retrieve textbook passages that answer it\nQuery: "
EMBEDDERS = {
    "qwen3-embedding:0.6b": dict(query_prefix=QWEN_INSTRUCT, doc_prefix=""),
    "bge-m3":               dict(query_prefix="", doc_prefix=""),
    "nomic-embed-text":     dict(query_prefix="search_query: ", doc_prefix="search_document: "),
}

f:\RP\Edu-Rag\rp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 · Loading chunks, BM25 and the eval set

In [2]:
def load_chunks():
    chunks = [json.loads(l) for l in CHUNKS_FILE.read_text(encoding="utf-8").splitlines()]
    return chunks, json.loads(CHUNK_SETTINGS_FILE.read_text())


def load_bm25():
    with open(BM25_FILE, "rb") as f:
        return pickle.load(f)


def load_questions():
    with open(QUESTIONS_FILE, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["answerable"] = r["answerable"].strip().lower() == "yes"
        r["gold_pages"] = [int(p) for p in r["gold_pages"].split(";") if p.strip()]
    return rows


chunks, chunk_settings = load_chunks()
by_id = {c["chunk_id"]: c for c in chunks}
bm = load_bm25()
assert bm["chunk_settings"] == chunk_settings["hash"], "BM25 is stale: re-run 03"
questions = load_questions()
print(len(chunks), "chunks;", len(questions), "questions:", Counter(q["type"] for q in questions),
      "| answerable", sum(q["answerable"] for q in questions), "| status", Counter(q["status"] for q in questions))

476 chunks; 68 questions: Counter({'conceptual': 24, 'definition': 17, 'code': 16, 'compare': 11}) | answerable 52 | status Counter({'VERIFIED': 68})


## 3 · Embedding and Chroma collections
Same embedding, cache and collection logic as 03, so that 04 can build the collections for the other embedders by itself. A collection is reused if it already matches the current chunks.

In [3]:
def model_slug(model):
    return re.sub(r"[^a-z0-9]+", "_", model.lower()).strip("_")


def embed_texts(texts, model, prefix="", batch=16):
    out = []
    for i in range(0, len(texts), batch):
        r = requests.post(f"{OLLAMA_URL}/api/embed", timeout=300,
                          json={"model": model, "input": [prefix + t for t in texts[i:i + batch]], "keep_alive": "5m"})
        r.raise_for_status()
        out += r.json()["embeddings"]
    v = np.asarray(out, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def unload(model):
    requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": [], "keep_alive": 0}, timeout=60)


def text_hash(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def embed_chunks(chunks, model, settings_hash):
    prefix, vectors, stats = EMBEDDERS[model]["doc_prefix"], {}, Counter()
    by_doc = defaultdict(list)
    for c in chunks:
        by_doc[c["doc_hash"]].append(c)
    for doc_hash, cs in by_doc.items():
        path = CACHE_DIR / model_slug(model) / f"{doc_hash[:16]}_{settings_hash}.npz"
        cached = {}
        if path.exists():
            z = np.load(path, allow_pickle=False)
            cached = {(i, h): v for i, h, v in zip(z["ids"], z["text_hashes"], z["vectors"])}
        hashes = [text_hash(c["text"]) for c in cs]
        todo = [(c, h) for c, h in zip(cs, hashes) if (c["chunk_id"], h) not in cached]
        if todo:
            cached.update({(c["chunk_id"], h): v for (c, h), v in zip(todo, embed_texts([c["text"] for c, _ in todo], model, prefix))})
        stats["embedded"] += len(todo)
        stats["from_cache"] += len(cs) - len(todo)
        vecs = np.stack([cached[(c["chunk_id"], h)] for c, h in zip(cs, hashes)])
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(path, ids=np.array([c["chunk_id"] for c in cs]), text_hashes=np.array(hashes), vectors=vecs)
        vectors.update({c["chunk_id"]: v for c, v in zip(cs, vecs)})
    return vectors, stats


META_KEYS = ["chunk_id", "source_id", "source_name", "role", "page_start", "page_end",
             "title", "has_code", "has_table", "n_tokens", "doc_hash"]
client = chromadb.PersistentClient(path=str(CHROMA_DIR))


def ensure_collection(model):
    """Return the model's collection, (re)building it if it doesn't match the current chunks."""
    name = f"chunks_{model_slug(model)}"
    if name in {c.name for c in client.list_collections()}:
        col = client.get_collection(name)
        if col.metadata.get("chunk_settings") == chunk_settings["hash"] and set(col.get(include=[])["ids"]) == set(by_id):
            return col, "reused"
        client.delete_collection(name)
    t0 = time.time()
    vectors, stats = embed_chunks(chunks, model, chunk_settings["hash"])
    col = client.create_collection(name, metadata={"hnsw:space": "cosine", "embed_model": model,
                                                   "chunk_settings": chunk_settings["hash"]})
    for i in range(0, len(chunks), 500):
        part = chunks[i:i + 500]
        col.add(ids=[c["chunk_id"] for c in part], embeddings=np.stack([vectors[c["chunk_id"]] for c in part]).tolist(),
                documents=[c["text"] for c in part], metadatas=[{k: c[k] for k in META_KEYS} for c in part])
    return col, f"built ({dict(stats)}, {time.time() - t0:.0f} s)"

## 4 · The three retrieval methods
Every method returns a ranked list of `(chunk_id, score)`, already restricted by the role/source filter.

In [4]:
def make_where(roles, sources):
    where = {"role": {"$in": list(roles)}}
    if sources:
        where = {"$and": [where, {"source_id": {"$in": list(sources)}}]}
    return where


def dense_search(col, qvec, n=CANDIDATES, roles=EVIDENCE_ROLES, sources=None):
    r = col.query(query_embeddings=[qvec.tolist()], n_results=n, where=make_where(roles, sources), include=["distances"])
    return [(i, 1 - d) for i, d in zip(r["ids"][0], r["distances"][0])]


def tokenize(text):
    STOP = TOKEN_STOPWORDS
    return [t for t in re.findall(r"[a-z0-9_]+", text.lower()) if t not in STOP]


TOKEN_STOPWORDS = set("""a an the of to in on for and or is are was were be been it its this that these those with as by at from
into than then so if not no do does did can could will would should may might what which who whom how why when where
there their they them we you your our i he she his her""".split())          # same list as 03


def bm25_search(query, n=CANDIDATES, roles=EVIDENCE_ROLES, sources=None):
    scores = bm["bm25"].get_scores(tokenize(query))
    out = []
    for i in np.argsort(-scores):
        if scores[i] <= 0:
            break
        c = by_id[bm["chunk_ids"][i]]
        if c["role"] in roles and (not sources or c["source_id"] in sources):
            out.append((c["chunk_id"], float(scores[i])))
            if len(out) == n:
                break
    return out


def rrf(*ranked_lists, k=RRF_K, n=CANDIDATES):
    """Reciprocal Rank Fusion: score = sum over lists of 1 / (k + rank)."""
    fused = defaultdict(float)
    for lst in ranked_lists:
        for rank, (cid, _) in enumerate(lst, start=1):
            fused[cid] += 1 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])[:n]

## 5 · Reranker
The cross-encoder returns a sigmoid score in 0–1 for each (question, chunk) pair. Scores are memoised, because the same pairs come up under several methods.

In [5]:
reranker = CrossEncoder(RERANKER, device="cpu")
_rerank_memo = {}


def rerank_scores(query, chunk_ids):
    todo = [cid for cid in chunk_ids if (query, cid) not in _rerank_memo]
    if todo:
        scores = reranker.predict([(query, by_id[cid]["text"]) for cid in todo], batch_size=16)
        _rerank_memo.update({(query, cid): float(s) for cid, s in zip(todo, scores)})
    return [_rerank_memo[(query, cid)] for cid in chunk_ids]


def retrieve(query, method, col=None, qvec=None, reorder=False, k=K, roles=EVIDENCE_ROLES, sources=None):
    """Return (top-k chunk ids, top rerank score over the candidate pool)."""
    if method == "dense":
        cands = dense_search(col, qvec, roles=roles, sources=sources)
    elif method == "bm25":
        cands = bm25_search(query, roles=roles, sources=sources)
    else:
        cands = rrf(dense_search(col, qvec, roles=roles, sources=sources), bm25_search(query, roles=roles, sources=sources))
    ids = [cid for cid, _ in cands]
    scores = rerank_scores(query, ids) if ids else []
    if reorder:
        ids = [cid for _, cid in sorted(zip(scores, ids), key=lambda x: -x[0])]
    return ids[:k], max(scores, default=0.0)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 499.53it/s]


## 6 · Metrics
A chunk is **relevant** when it comes from the gold source and its page range overlaps a gold page.
- **Recall@5:** share of the gold pages covered by the top 5.
- **Hit@5:** at least one relevant chunk in the top 5 (shown too, since one good slide is often enough to answer).
- **MRR:** 1 / rank of the first relevant chunk (0 if none in the top 20).
- **nDCG@5:** binary relevance, ideal = as many relevant chunks as exist, capped at 5.

In [6]:
def is_relevant(cid, q):
    c = by_id[cid]
    return c["source_name"] == q["gold_source"] and any(c["page_start"] <= p <= c["page_end"] for p in q["gold_pages"])


def n_relevant_chunks(q):
    return sum(is_relevant(cid, q) for cid in by_id)


def metrics_for(ranked, q, k=K):
    top = ranked[:k]
    covered = {p for cid in top for p in q["gold_pages"]
               if by_id[cid]["source_name"] == q["gold_source"] and by_id[cid]["page_start"] <= p <= by_id[cid]["page_end"]}
    first = next((i for i, cid in enumerate(ranked, start=1) if is_relevant(cid, q)), None)
    dcg = sum(1 / math.log2(i + 1) for i, cid in enumerate(top, start=1) if is_relevant(cid, q))
    idcg = sum(1 / math.log2(i + 1) for i in range(1, min(n_relevant_chunks(q), k) + 1))
    return dict(recall=len(covered) / len(q["gold_pages"]), hit=float(any(is_relevant(c, q) for c in top)),
                mrr=1 / first if first else 0.0, ndcg=dcg / idcg if idcg else 0.0)


def mean_metrics(rows):
    return {m: float(np.mean([r[m] for r in rows])) for m in ("recall", "hit", "mrr", "ndcg")} if rows else {}

## 7 · Run the evaluation
For each embedder: build or reuse its collection, embed all questions at once, run dense / BM25 / hybrid with reordering off and on, then unload the model. MRR uses the full 20-candidate list; the other metrics use the top 5.

In [7]:
def evaluate(questions):
    answerable = [q for q in questions if q["answerable"]]
    results, per_question, top_scores, build_notes = {}, {}, defaultdict(dict), {}
    for model in EMBEDDERS:
        col, note = ensure_collection(model)
        build_notes[model] = note
        qvecs = embed_texts([q["question"] for q in questions], model, EMBEDDERS[model]["query_prefix"])
        unload(model)
        qvec_of = {q["question"]: v for q, v in zip(questions, qvecs)}
        for method in ("dense", "bm25", "hybrid"):
            if method == "bm25" and model != next(iter(EMBEDDERS)):
                continue                                   # BM25 doesn't depend on the embedder
            for reorder in (False, True):
                name = f"{method}{'+rerank' if reorder else ''}" + ("" if method == "bm25" else f" | {model}")
                rows = []
                for q in questions:
                    full, top = retrieve(q["question"], method, col, qvec_of[q["question"]], reorder=reorder, k=CANDIDATES)
                    top_scores[name][q["question"]] = top
                    if q["answerable"]:
                        rows.append(dict(metrics_for(full, q), type=q["type"], question=q["question"], top5=full[:K]))
                results[name] = rows
    return results, top_scores, build_notes


t0 = time.time()
results, top_scores, build_notes = evaluate(questions)
print(f"evaluated {len(results)} configurations in {time.time() - t0:.0f} s")
for m, n in build_notes.items():
    print(f"  {m}: collection {n}")

evaluated 14 configurations in 862 s
  qwen3-embedding:0.6b: collection reused
  bge-m3: collection reused
  nomic-embed-text: collection reused


---
# Checks

## 8 · Eval set sanity: gold pages exist, unanswerable topics are absent

In [8]:
missing = [(q["question"], p) for q in questions if q["answerable"] for p in q["gold_pages"]
           if not any(c["source_name"] == q["gold_source"] and c["page_start"] <= p <= c["page_end"] for c in chunks)]
print("gold pages with no chunk:", missing or "none")
print("relevant chunks per answerable question:", Counter(n_relevant_chunks(q) for q in questions if q["answerable"]))
KEY_TERMS = {"Apriori": "apriori", "fuzzy": "fuzzy", "crossover": "crossover", "convolutional": "convolution",
             "backpropagation": "backprop", "attention": "attention", "inherits": "inherit", "MySQL": "mysql",
             "TCP": r"\btcp\b", "primary key": "primary key", "Markov": "markov", "perplexity": "perplexity",
             "dropout": "dropout", "momentum": "momentum", "Python set": r"\bset\(|\.add\(",
             "gradient boosting": "gradient boosting"}
for q in questions:
    if not q["answerable"]:
        term = next(v for k, v in KEY_TERMS.items() if k in q["question"])
        n = sum(bool(re.search(term, c["text"], re.I)) for c in chunks)
        print(f"  unanswerable, '{term}' in {n} chunks: {q['question']}")

gold pages with no chunk: none
relevant chunks per answerable question: Counter({1: 28, 2: 19, 3: 4, 4: 1})
  unanswerable, 'apriori' in 0 chunks: What is the Apriori algorithm?
  unanswerable, 'fuzzy' in 0 chunks: What is a fuzzy membership function?
  unanswerable, 'crossover' in 0 chunks: What is the crossover operator in a genetic algorithm?
  unanswerable, 'convolution' in 0 chunks: How does a convolutional neural network work?
  unanswerable, 'backprop' in 0 chunks: How does backpropagation compute gradients?
  unanswerable, 'attention' in 0 chunks: How does the attention mechanism in transformers work?
  unanswerable, 'inherit' in 0 chunks: How do I create a class that inherits from another class in Python?
  unanswerable, 'mysql' in 0 chunks: How do I connect to a MySQL database from Python?
  unanswerable, '\btcp\b' in 0 chunks: What is the difference between TCP and UDP?
  unanswerable, 'primary key' in 0 chunks: What is the difference between a primary key and a foreign key?

## 9 · Main table: every method and embedder (all answerable questions)

In [9]:
def table(results):
    rows = sorted(((name, mean_metrics(r)) for name, r in results.items()), key=lambda x: -x[1]["ndcg"])
    print(f"{'configuration':52} {'Recall@5':>8} {'Hit@5':>6} {'MRR':>6} {'nDCG@5':>7}")
    for name, m in rows:
        print(f"{name:52} {m['recall']:8.3f} {m['hit']:6.3f} {m['mrr']:6.3f} {m['ndcg']:7.3f}")
    return rows


ranking = table(results)
best_name = ranking[0][0]
print("\nbest by nDCG@5:", best_name)

configuration                                        Recall@5  Hit@5    MRR  nDCG@5
dense | bge-m3                                          0.880  0.923  0.829   0.809
dense | nomic-embed-text                                0.870  0.923  0.799   0.786
dense | qwen3-embedding:0.6b                            0.893  0.962  0.757   0.775
hybrid | nomic-embed-text                               0.840  0.923  0.801   0.753
dense+rerank | qwen3-embedding:0.6b                     0.875  0.923  0.750   0.748
hybrid | qwen3-embedding:0.6b                           0.867  0.962  0.760   0.747
hybrid | bge-m3                                         0.849  0.942  0.771   0.746
hybrid+rerank | qwen3-embedding:0.6b                    0.875  0.923  0.722   0.727
dense+rerank | bge-m3                                   0.865  0.923  0.725   0.724
hybrid+rerank | bge-m3                                  0.856  0.904  0.721   0.718
hybrid+rerank | nomic-embed-text                        0.861  0.923  0.724 

## 10 · Per question type (Hit@5 / nDCG@5)

In [10]:
TYPES = ["definition", "conceptual", "code", "compare"]
counts = Counter(q["type"] for q in questions if q["answerable"])
print(f"{'configuration':52} " + " ".join(f"{t + ' (' + str(counts[t]) + ')':>17}" for t in TYPES))
for name, _ in ranking:
    cells = []
    for t in TYPES:
        m = mean_metrics([r for r in results[name] if r["type"] == t])
        cells.append(f"{m['hit']:.2f} / {m['ndcg']:.2f}".rjust(17))
    print(f"{name:52} " + " ".join(cells))

configuration                                          definition (13)   conceptual (19)         code (12)       compare (8)
dense | bge-m3                                             1.00 / 0.88       0.89 / 0.76       0.83 / 0.83       1.00 / 0.80
dense | nomic-embed-text                                   1.00 / 0.84       0.89 / 0.72       0.92 / 0.84       0.88 / 0.76
dense | qwen3-embedding:0.6b                               1.00 / 0.82       0.95 / 0.66       0.92 / 0.82       1.00 / 0.90
hybrid | nomic-embed-text                                  1.00 / 0.79       0.89 / 0.66       0.92 / 0.86       0.88 / 0.77
dense+rerank | qwen3-embedding:0.6b                        1.00 / 0.79       0.89 / 0.69       0.92 / 0.80       0.88 / 0.75
hybrid | qwen3-embedding:0.6b                              1.00 / 0.76       0.95 / 0.64       0.92 / 0.86       1.00 / 0.82
hybrid | bge-m3                                            1.00 / 0.76       0.95 / 0.65       0.92 / 0.85       0.88 / 0.80


## 10b · Easy vs hard questions
Easy = the first 30, which mostly reuse slide-title words. Hard = reworded questions that avoid them.

In [11]:
diff_of = {q["question"]: q.get("difficulty", "easy") for q in questions}
n_diff = Counter(diff_of[q["question"]] for q in questions if q["answerable"])
print(f"{'configuration':52} {'easy (' + str(n_diff['easy']) + ') Hit/nDCG/MRR':>26} {'hard (' + str(n_diff['hard']) + ') Hit/nDCG/MRR':>26}")
for name, _ in ranking:
    cells = []
    for d in ("easy", "hard"):
        m = mean_metrics([r for r in results[name] if diff_of[r["question"]] == d])
        cells.append(f"{m['hit']:.2f} / {m['ndcg']:.2f} / {m['mrr']:.2f}".rjust(26))
    print(f"{name:52} " + " ".join(cells))

configuration                                            easy (30) Hit/nDCG/MRR     hard (22) Hit/nDCG/MRR
dense | bge-m3                                               1.00 / 0.89 / 0.91         0.82 / 0.70 / 0.72
dense | nomic-embed-text                                     1.00 / 0.89 / 0.91         0.82 / 0.64 / 0.65
dense | qwen3-embedding:0.6b                                 1.00 / 0.85 / 0.83         0.91 / 0.67 / 0.65
hybrid | nomic-embed-text                                    0.97 / 0.84 / 0.88         0.86 / 0.64 / 0.69
dense+rerank | qwen3-embedding:0.6b                          1.00 / 0.84 / 0.83         0.82 / 0.62 / 0.64
hybrid | qwen3-embedding:0.6b                                0.97 / 0.81 / 0.82         0.95 / 0.66 / 0.68
hybrid | bge-m3                                              1.00 / 0.83 / 0.85         0.86 / 0.63 / 0.66
hybrid+rerank | qwen3-embedding:0.6b                         1.00 / 0.84 / 0.83         0.82 / 0.57 / 0.57
dense+rerank | bge-m3                

## 11 · Does reordering by the reranker help?

In [12]:
for name in results:
    if "+rerank" in name:
        continue
    base, rr = mean_metrics(results[name]), mean_metrics(results[name.replace(" |", "+rerank |") if "|" in name else name + "+rerank"])
    print(f"{name:42} nDCG@5 {base['ndcg']:.3f} -> {rr['ndcg']:.3f} ({rr['ndcg'] - base['ndcg']:+.3f}) | "
          f"MRR {base['mrr']:.3f} -> {rr['mrr']:.3f} ({rr['mrr'] - base['mrr']:+.3f})")

dense | qwen3-embedding:0.6b               nDCG@5 0.775 -> 0.748 (-0.027) | MRR 0.757 -> 0.750 (-0.006)
bm25                                       nDCG@5 0.665 -> 0.711 (+0.046) | MRR 0.710 -> 0.726 (+0.017)
hybrid | qwen3-embedding:0.6b              nDCG@5 0.747 -> 0.727 (-0.020) | MRR 0.760 -> 0.722 (-0.038)
dense | bge-m3                             nDCG@5 0.809 -> 0.724 (-0.085) | MRR 0.829 -> 0.725 (-0.104)
hybrid | bge-m3                            nDCG@5 0.746 -> 0.718 (-0.028) | MRR 0.771 -> 0.721 (-0.050)
dense | nomic-embed-text                   nDCG@5 0.786 -> 0.707 (-0.079) | MRR 0.799 -> 0.706 (-0.093)
hybrid | nomic-embed-text                  nDCG@5 0.753 -> 0.713 (-0.040) | MRR 0.801 -> 0.724 (-0.077)


## 12 · Abstain threshold from the reranker score
For each question, the highest reranker score in the candidate pool of the best configuration. Answerable questions should score high, unanswerable ones low. The threshold is the one that maximises balanced accuracy (mean of "answers when it should" and "abstains when it should").

In [13]:
def choose_threshold(scores, questions):
    ans = np.array([scores[q["question"]] for q in questions if q["answerable"]])
    una = np.array([scores[q["question"]] for q in questions if not q["answerable"]])
    best = None
    for t in sorted(set(np.concatenate([ans, una]).tolist())):
        tpr, tnr = (ans >= t).mean(), (una < t).mean()
        cand = ((tpr + tnr) / 2, -abs(tpr - tnr), t, tpr, tnr)
        best = max(best, cand) if best else cand
    bal, _, t, tpr, tnr = best
    return dict(threshold=float(t), balanced_accuracy=float(bal), answers_answerable=float(tpr),
                abstains_unanswerable=float(tnr), answerable_scores=ans, unanswerable_scores=una)


abst = choose_threshold(top_scores[best_name], questions)
fmt = lambda a: " ".join(f"{x:.3f}" for x in sorted(a))
print("configuration:", best_name)
print("answerable   :", fmt(abst["answerable_scores"]))
print("unanswerable :", fmt(abst["unanswerable_scores"]))
print(f"\nthreshold {abst['threshold']:.3f}: answers {abst['answers_answerable']:.0%} of answerable, "
      f"abstains on {abst['abstains_unanswerable']:.0%} of unanswerable (balanced accuracy {abst['balanced_accuracy']:.2f})")
for q in questions:
    s = top_scores[best_name][q["question"]]
    wrong = (s >= abst["threshold"]) != q["answerable"]
    if wrong:
        print(f"  misjudged ({'answerable' if q['answerable'] else 'unanswerable'}, score {s:.3f}): {q['question']}")

configuration: dense | bge-m3
answerable   : 0.004 0.006 0.008 0.012 0.013 0.024 0.046 0.049 0.052 0.062 0.079 0.092 0.104 0.229 0.261 0.458 0.502 0.548 0.685 0.687 0.832 0.838 0.869 0.873 0.907 0.912 0.928 0.932 0.939 0.940 0.942 0.957 0.965 0.974 0.974 0.976 0.982 0.984 0.987 0.992 0.993 0.995 0.995 0.996 0.997 0.998 0.999 1.000 1.000 1.000 1.000 1.000
unanswerable : 0.001 0.002 0.002 0.006 0.015 0.027 0.034 0.065 0.067 0.079 0.161 0.188 0.254 0.836 0.992 0.998

threshold 0.261: answers 73% of answerable, abstains on 81% of unanswerable (balanced accuracy 0.77)
  misjudged (unanswerable, score 0.992): How does a convolutional neural network work?
  misjudged (unanswerable, score 0.998): What is the difference between TCP and UDP?
  misjudged (answerable, score 0.013): Why does changing one list also change another list I assigned it to?
  misjudged (answerable, score 0.006): Why can't I replace a single character inside a string directly?
  misjudged (answerable, score 0.049): My mod

## 13 · Misses of the best configuration

In [14]:
def show_misses(name):
    for r in results[name]:
        if r["hit"] == 0:
            q = next(q for q in questions if q["question"] == r["question"])
            print(f"MISS [{r['type']}] {r['question']}  (gold {q['gold_source'][:30]} p{q['gold_pages']})")
            for cid in r["top5"][:3]:
                c = by_id[cid]
                print(f"      got {c['source_name'][:30]:30} p{c['page_start']}-{c['page_end']} {c['title'][:45]}")


show_misses(best_name)

MISS [code] Why does changing one list also change another list I assigned it to?  (gold Ch1 and 2 Introduction to Pyth p[78])
      got Ch1 and 2 Introduction to Pyth p68-71 Lists
      got Ch1 and 2 Introduction to Pyth p73-74 Lists operation
      got Ch3 Functions Modules and Pack p15-15 passing by reference
MISS [conceptual] How do I measure how well each point sits in its own group compared with the next closest group?  (gold Ch 4 Unsupervised Learning.pdf p[32, 33, 34])
      got Ch 4 Unsupervised Learning.pdf p51-51 DBSCAN
      got Ch 4 Unsupervised Learning.pdf p53-53 DBSCAN
      got Ch 4 Unsupervised Learning.pdf p41-41 Agglomerative Hierarchical
MISS [code] How do I get just the part of a matched pattern that was inside brackets?  (gold Ch 7. Regular Expressions.pdf p[17, 18])
      got Ch 7. Regular Expressions.pdf  p8-8 What’s a match object?
      got Ch 7. Regular Expressions.pdf  p19-19 Examples
      got Ch 7. Regular Expressions.pdf  p4-5 Match()
MISS [conceptual] H

## 14 · Save results

In [15]:
summary = {name: dict(mean_metrics(rows), per_type={t: mean_metrics([r for r in rows if r["type"] == t]) for t in TYPES})
           for name, rows in results.items()}
(EVAL_DIR / "retrieval_results.json").write_text(json.dumps(dict(
    best=best_name, k=K, candidates=CANDIDATES, rrf_k=RRF_K, chunk_settings=chunk_settings["hash"],
    n_questions=len(questions), summary=summary), indent=2))
(EVAL_DIR / "abstain_threshold.json").write_text(json.dumps(dict(
    configuration=best_name, reranker=RERANKER, threshold=abst["threshold"],
    balanced_accuracy=abst["balanced_accuracy"], answers_answerable=abst["answers_answerable"],
    abstains_unanswerable=abst["abstains_unanswerable"], eval_status=dict(Counter(q["status"] for q in questions))), indent=2))
print("saved:", [p.name for p in EVAL_DIR.glob("*.json")])
print(requests.get(f"{OLLAMA_URL}/api/ps").json()["models"] or "no Ollama models loaded")

saved: ['abstain_rule.json', 'abstain_threshold.json', 'generation_summary.json', 'retrieval_results.json', 'router_cases.json', 'router_results.json']
[{'name': 'qwen2.5:7b-instruct', 'model': 'qwen2.5:7b-instruct', 'size': 4987132312, 'digest': '845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '7.6B', 'quantization_level': 'Q4_K_M'}, 'expires_at': '2026-10-25T21:33:19.6852335+05:30', 'size_vram': 4987132312, 'context_length': 8192}, {'name': 'bge-m3:latest', 'model': 'bge-m3:latest', 'size': 664000265, 'digest': '7907646426070047a77226ac3e684fbbe8410524f7b4a74d02837e43f2146bab', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'bert', 'families': ['bert'], 'parameter_size': '566.70M', 'quantization_level': 'F16'}, 'expires_at': '2026-10-25T21:33:16.1378336+05:30', 'size_vram': 664000265, 'context_length': 4096}]
